# Task 12-1

Построение изображения предобученным генератором и вычисление средней интенсивности.

Шаги:
- загрузить генератор и тензор входа;
- сгенерировать изображение;
- выполнить MinMax-преобразование в диапазон `[0, 255]`;
- округлить до целых по математическим правилам;
- перевести в шкалу, где белый = `255`, черный = `0`;
- вывести среднюю интенсивность, округленную до тысячных.

In [3]:
from pathlib import Path
import urllib.request

import numpy as np
import torch
from torch import nn


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 784),
            nn.Tanh(),
        )

    def forward(self, x):
        output = self.model(x)
        return output.view(x.size(0), 1, 28, 28)


BASE_DIR = Path.cwd()
if BASE_DIR.name != "task-12-1" and (BASE_DIR / "task-12-1").exists():
    BASE_DIR = BASE_DIR / "task-12-1"

generator_path = BASE_DIR / "generator_20"
tensor_path = BASE_DIR / "tensor.pt"

generator_url = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_6_1/generator_20"
tensor_url = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_6_1/task_1_test_files/Comp_Vision_Task_6_Test_file_12.pt"

if not generator_path.exists():
    print("Скачиваю generator_20...")
    urllib.request.urlretrieve(generator_url, generator_path)

if not tensor_path.exists():
    print("Скачиваю tensor.pt...")
    urllib.request.urlretrieve(tensor_url, tensor_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    generator = torch.load(generator_path, map_location=device, weights_only=False)
except Exception:
    generator = Generator().to(device)
    state = torch.load(generator_path, map_location=device)
    generator.load_state_dict(state)

generator.eval()

try:
    z = torch.load(tensor_path, map_location=device)
except Exception as e:
    raise RuntimeError(
        "Не удалось загрузить tensor.pt как PyTorch-тензор. "
        "Проверьте, что это корректный .pt файл из задания."
    ) from e

if z.ndim == 1:
    z = z.unsqueeze(0)

with torch.no_grad():
    generated = generator(z).cpu().numpy()

img = generated[0, 0]

img_min = img.min()
img_max = img.max()
img_255 = (img - img_min) / (img_max - img_min) * 255.0

# Округление до ближайшего целого по математическим правилам.
img_255 = np.floor(img_255 + 0.5).clip(0, 255).astype(np.uint8)

# Приводим к соглашению: белый=255, черный=0.
img_255 = 255 - img_255

mean_intensity = round(float(img_255.mean()), 3)
print(mean_intensity)

180.89
